In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data
import numpy as np 
import sys
sys.path.append('..')
from envs.routing_env import RoutingEnv

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


In [2]:
class GNNDQNPolicy(nn.Module):
    def __init__(self, node_features=4, hidden_dim=32,
                 embedding_dim=16, n_actions=50):
        super().__init__()
        # GCN Encoder
        self.conv1 = GCNConv(node_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, embedding_dim)
        # DQN Head
        self.fc1 = nn.Linear(embedding_dim, 64)
        self.fc2 = nn.Linear(64, n_actions)

    def forward(self, x, edge_index, batch=None):
        # GCN encoding
        h = F.relu(self.conv1(x, edge_index))
        h = F.relu(self.conv2(h, edge_index))
        # Global mean pooling
        if batch is None:
            batch = torch.zeros(x.shape[0], dtype=torch.long)
        graph_embed = global_mean_pool(h, batch)
        # Q-values
        q = F.relu(self.fc1(graph_embed))
        q = self.fc2(q)
        return q

# Test forward pass
policy = GNNDQNPolicy()
total_params = sum(p.numel() for p in policy.parameters())
print("Total parameters:", total_params)

# Load graph
data = torch.load('../results/graph_data.pt', weights_only=False)
q_values = policy(data.x, data.edge_index)
print("Q-values shape:", q_values.shape)
print("Q-values (first 10):", q_values[0, :10].detach().numpy())
print("Policy forward pass OK!")

Total parameters: 5026
Q-values shape: torch.Size([1, 50])
Q-values (first 10): [-0.18169942 -0.15405528 -0.06196889 -0.08061136  0.03958007  0.01737002
 -0.00694694 -0.09458438 -0.00152038  0.15375814]
Policy forward pass OK!


In [3]:
env = RoutingEnv()
obs, _ = env.reset()
print("Observation shape:", obs.shape)
print("Action space:", env.action_space.n)
print("Env and policy compatible!")

Observation shape: (8,)
Action space: 50
Env and policy compatible!
